# Polars Conversion Meeting 1
This notebook is a guide for how we will translate the `datascience` and `numpy` libraries to `polars` for Data 8: Foundations of Data Science


Date: 9/22/2026

Attendees: John DeNero, Rinrada Maneenop, Brandon Su, Lisa Yan

In [ ]:
%pip install -q \
    pyarrow \
    polars==2.1.0 \
    hvplot==0.12.2

In [1]:
from datascience import *
import numpy as np
import polars as pl
import seaborn as sns
%matplotlib inline
import matplotlib.pyplot as plt

## Arrays vs Series

### Declaration

In [2]:
# numpy
np.array([1, 2, 3])

array([1, 2, 3])

In [3]:
# datascience
make_array(1, 2, 3)

array([1, 2, 3])

In [4]:
# polars
pl.Series('x', [1,2,3]) # Although it's not required, we'll always include a name for the Series (like 'x')

x
i64
1
2
3


### Arithmetic

In [5]:
# datascience
3 * make_array(1, 2, 3) + 4

array([ 7, 10, 13])

In [6]:
# polars
3 * pl.Series('x', [1, 2, 3]) + 4

x
i64
7
10
13


### Element-Wise Operations

In [7]:
# datascience
make_array(1, 2, 3) + make_array(1, 10, 100)

array([  2,  12, 103])

In [8]:
# polars
pl.Series('x', [1, 2, 3]) + pl.Series('y', [1, 10, 100]) # Observe that the Series name is based off the first Series

x
i64
2
12
103


## For Loops

In [9]:
# datascience
for i in np.arange(10):
    ...

In [10]:
# polars 
for i in range(10):
    ...

## `.with_column` - datascience vs polars

### datascience

In [11]:
ds_table = Table().with_columns(
    "foo", make_array(1, 2),
    "bar", make_array(3, 4)
)
ds_table

foo,bar
1,3
2,4


In [12]:
ds_table.with_columns(
    "baz", ds_table.column("foo") + ds_table.column("bar")
)

foo,bar,baz
1,3,4
2,4,6


### polars

In [13]:
pl_table = pl.DataFrame().with_columns(
    pl.Series("foo", [1, 2]),
    pl.Series("bar", [3, 4])
)
pl_table

foo,bar
i64,i64
1,3
2,4


In [14]:
pl_table.with_columns(
    (pl.col('foo') + pl.col('bar')).alias('baz')
)

foo,bar,baz
i64,i64,i64
1,3,4
2,4,6


In [15]:
# This cell includes additional ways to add a new column to a dataframe. 
# Ignore these other approaches and just reference the example cell above.

# Option 1
pl_table.with_columns(
    baz=pl_table["foo"] + pl_table["bar"]
)

# Option 2
pl_table.with_columns(
    pl.Series('baz', pl_table['foo'] + pl_table['bar'])
)

# Option 3
pl_table.with_columns(
    baz=pl.col('foo') + pl.col('bar')
)

foo,bar,baz
i64,i64,i64
1,3,4
2,4,6


## Filtering

In [16]:
cones_ds = Table().with_columns(
    'Flavor', ['strawberry', 'chocolate', 'chocolate', 'strawberry', 'chocolate', 'bubblegum'],
    'Color', ['pink', 'light brown', 'dark brown', 'pink', 'dark brown', 'pink'],
    'Price', [3.55, 4.75, 5.25, 5.25, 5.25, 4.75]
)


cones_pl = pl.DataFrame().with_columns(
    pl.Series('Flavor', ['strawberry', 'chocolate', 'chocolate', 'strawberry', 'chocolate', 'bubblegum']),
    pl.Series('Color', ['pink', 'light brown', 'dark brown', 'pink', 'dark brown', 'pink']),
    pl.Series('Price', [3.55, 4.75, 5.25, 5.25, 5.25, 4.75])
)
cones_pl

Flavor,Color,Price
str,str,f64
"""strawberry""","""pink""",3.55
"""chocolate""","""light brown""",4.75
"""chocolate""","""dark brown""",5.25
"""strawberry""","""pink""",5.25
"""chocolate""","""dark brown""",5.25
"""bubblegum""","""pink""",4.75


In [17]:
# datascience
cones_ds.where("Price", are.above(5))

Flavor,Color,Price
chocolate,dark brown,5.25
strawberry,pink,5.25
chocolate,dark brown,5.25


In [18]:
# polars
cones_pl.filter(pl.col("Price") > 5)

# Note: cones_pl.filter(cones_pl['Price'] > 5) works but we'll avoid this

Flavor,Color,Price
str,str,f64
"""chocolate""","""dark brown""",5.25
"""strawberry""","""pink""",5.25
"""chocolate""","""dark brown""",5.25


## Grouping

### datascience

In [19]:
cones_ds.group("Flavor", max)

Flavor,Color max,Price max
bubblegum,pink,4.75
chocolate,light brown,5.25
strawberry,pink,5.25


### polars

In [20]:
cones_pl.group_by(pl.col("Flavor")).max()

Flavor,Color,Price
str,str,f64
"""chocolate""","""light brown""",5.25
"""bubblegum""","""pink""",4.75
"""strawberry""","""pink""",5.25


In [21]:
# Note: Be sure to check that the function can be applied to all columns. 
# For example, `cones_pl.group_by(pl.col("Flavor")).sum()` will error since you can't sum strings.
# But `cones_pl.select("Flavor", "Price").group_by(pl.col('Flavor')).sum()` works OK.

In [22]:
cones_pl.select("Flavor", "Price").group_by(pl.col('Flavor')).sum()

Flavor,Price
str,f64
"""chocolate""",15.25
"""bubblegum""",4.75
"""strawberry""",8.8


## Pivot

In [23]:
# datascience
cones_ds.pivot("Flavor", "Color", "Price", max)

Color,bubblegum,chocolate,strawberry
dark brown,0,5.25,0
light brown,0,4.75,0
pink,4.75,0,5.25


In [24]:
# polars
cones_pl.pivot("Flavor", index="Color", values="Price", aggregate_function="max")

Color,strawberry,chocolate,bubblegum
str,f64,f64,f64
"""pink""",5.25,null,4.75
"""light brown""",null,4.75,null
"""dark brown""",null,5.25,null


## Lecture Example

### datascience

In [25]:
streets_ds = make_array('Bancroft', 'Durant', 'Channing', 'Haste')
# streets
southside_ds = Table().with_column('Street', streets_ds)
# southside
southside_ds.with_column('Blocks from campus', np.arange(4))

# how might I get "Channing" from this table?
southside_ds = southside_ds.with_column('Blocks from campus', np.arange(4))
southside_ds.where("Street", "Channing").column("Street").item(0)

'Channing'

### polars

In [26]:
streets_pl = pl.Series('Street', ['Bancroft', 'Durant', 'Channing', 'Haste'])
streets_pl

Street
str
"""Bancroft"""
"""Durant"""
"""Channing"""
"""Haste"""


In [27]:
southside_pl = pl.DataFrame().with_columns(streets_pl)
southside_pl

Street
str
"""Bancroft"""
"""Durant"""
"""Channing"""
"""Haste"""


In [28]:
southside_pl = southside_pl.with_columns(pl.Series("Blocks from campus", range(4)))
southside_pl

Street,Blocks from campus
str,i64
"""Bancroft""",0
"""Durant""",1
"""Channing""",2
"""Haste""",3


In [29]:
southside_pl.filter(pl.col("Street") == "Channing").get_column('Street')

Street
str
"""Channing"""


In [30]:
southside_pl.filter(pl.col("Street") == "Channing").get_column('Street').item(0)

'Channing'